In [11]:
import copy
import time
import math
import random
import os

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader
from torchvision.utils import make_grid
from scipy import linalg
from sklearn.model_selection import train_test_split

In [4]:
SEED    = 42
BATCH   = 128
ZDIM    = 64           # latent size of AE / Sparse AE / VAE / MLP feature
NZ      = 100          # noise size of GAN / DCGAN
LAMBDA  = 5e-5         # sparsity weight (Sparse AE)
T       = 1000         # number of diffusion steps
N_FID   = 5000         # number of generated images used for FID


EPOCHS = dict(
    mlp=40,
    dec=40,
    ae=40,
    sae=40,
    vae=40,
    gan=40,
    dcgan=40,
    ddpm=40,
)

PATIENCE = dict(
    mlp=5,
    dec=5,
    ae=5,
    sae=5,
    vae=5,
    ddpm=10,
)

MIN_DELTA = 1e-3 # minimum improvement for early stopping

In [ ]:
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def seed_all():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)


seed_all()

# 1. DATA

In [18]:
train_raw = torchvision.datasets.FashionMNIST(root="data", train=True, download=True)
test_raw = torchvision.datasets.FashionMNIST(root="data", train=False, download=True)

In [19]:
X = torch.cat([train_raw.data, test_raw.data])
X = X.float() / 255
X = X.unsqueeze(1) # shape (70000, 1, 28, 28)
Y = torch.cat([train_raw.targets, test_raw.targets])

In [20]:
X_train, X_temp, Y_train, Y_temp = train_test_split(
    X, Y,
    test_size=0.2,
    random_state=SEED,
    stratify=Y
)

X_val, X_test, Y_val, Y_test = train_test_split(
    X_temp, Y_temp,
    test_size=0.5,
    random_state=SEED,
    stratify=Y_temp
)

In [21]:
train_loader = DataLoader(TensorDataset(X_train, Y_train), BATCH, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, Y_val), BATCH, shuffle=False)
test_loader = DataLoader(TensorDataset(X_test, Y_test), BATCH, shuffle=False)

print("train", len(X_train), "| val", len(X_val), "| test", len(X_test))

train 56000 | val 7000 | test 7000


# 2. METRICS

In [ ]:
from torchmetrics import MeanSquaredError
from torchmetrics.image import (
    PeakSignalNoiseRatio,
    StructuralSimilarityIndexMeasure,
    FrechetInceptionDistance,
)

mse_metric = MeanSquaredError().to(dev)
psnr_metric = PeakSignalNoiseRatio(data_range=1.0, dim=(1, 2, 3)).to(dev)
ssim_metric = StructuralSimilarityIndexMeasure(data_range=1.0).to(dev)
fid_metric = FrechetInceptionDistance(
    feature=2048,
    reset_real_features=False,
    normalize=True,
).to(dev)

@torch.no_grad()
def recon_metrics(recon_fn):
    """Test MSE / PSNR / SSIM between recon_fn(x) and the original x."""
    mse_metric.reset()
    psnr_metric.reset()
    ssim_metric.reset()

    for x, _ in test_loader:
        x = x.to(dev)
        x_rec = recon_fn(x).clamp(0, 1)

        mse_metric.update(x_rec, x)
        psnr_metric.update(x_rec, x)
        ssim_metric.update(x_rec, x)

    return dict(
        mse=mse_metric.compute().item(),
        psnr=psnr_metric.compute().item(),
        ssim=ssim_metric.compute().item(),
    )


@torch.no_grad()
def add_images_to_fid(images, real):
    """Feed images (N, 1, 28, 28) in 0..1 to the FID metric. real=True for real images."""
    for start in range(0, len(images), 250):
        batch = images[start:start + 250].to(dev).clamp(0, 1)
        batch = batch.repeat(1, 3, 1, 1) 
        fid_metric.update(batch, real=real)


def fid_of(fake_images):
    fid_metric.reset()
    add_images_to_fid(fake_images, real=False)
    return fid_metric.compute().item()


def latency_ms(fn, runs=20):
    """Average time in milliseconds for one call of fn() (= one batch of 64 images)."""
    with torch.no_grad():
        fn()                                       # warm-up
        torch.cuda.synchronize()
        start = time.perf_counter()
        for _ in range(runs):
            fn()
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - start
    return elapsed / runs * 1000

# 3. MODELS

In [ ]:
from torchvision.ops import MLP
from diffusers import UNet2DModel, DDPMScheduler
from functools import partial

class MLPClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 256)
        self.fc2 = nn.Linear(256, 64)
        self.fc3 = nn.Linear(64, 10)

    def features(self, x):
        x = x.flatten(1)
        h = F.relu(self.fc1(x))
        z = F.relu(self.fc2(h))
        return z

    def forward(self, x):
        z = self.features(x)
        return self.fc3(z)

class AE(nn.Module):
    def __init__(self, lam=0.0):
        super().__init__()
        self.lam = lam

        self.enc1 = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 256),
            nn.ReLU(),
        )
        self.enc2 = nn.Linear(256, ZDIM)
        self.dec = MLP(ZDIM, [256, 784])

    def encode(self, x):
        h = self.enc1(x)
        z = self.enc2(h)
        return z

    def decode(self, z):
        pixels = torch.sigmoid(self.dec(z))
        return pixels.view(-1, 1, 28, 28)

    def recon(self, x):
        z = self.encode(x)
        return self.decode(z)

    def loss(self, x):
        h = self.enc1(x)
        z = self.enc2(h)
        x_rec = self.decode(z)

        recon_loss = F.mse_loss(x_rec, x)
        sparsity_loss = h.abs().sum(dim=1).mean()
        return recon_loss + self.lam * sparsity_loss


class VAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = MLP(784, [512, 256, 2 * ZDIM])
        self.decoder = MLP(ZDIM, [256, 512, 784])

    def encode(self, x):
        output = self.encoder(x.flatten(1))
        mu, logvar = output.chunk(2, dim=1)
        return mu, logvar

    def decode(self, z):
        pixels = torch.sigmoid(self.decoder(z))
        return pixels.view(-1, 1, 28, 28)

    def recon(self, x):
        mu, _ = self.encode(x)
        return self.decode(mu)

    def loss(self, x):
        mu, logvar = self.encode(x)

        std = torch.exp(0.5 * logvar)
        z = mu + std * torch.randn_like(mu)

        logits = self.decoder(z).view(-1, 1, 28, 28)
        recon_loss = F.binary_cross_entropy_with_logits(logits, x, reduction="sum") / len(x)
        kl_loss = -0.5 * torch.sum(1 + logvar - mu ** 2 - logvar.exp()) / len(x)
        return recon_loss + kl_loss

leaky_relu = partial(nn.LeakyReLU, 0.2)

G_gan = nn.Sequential(
    MLP(NZ, [256, 512, 1024, 784], activation_layer=leaky_relu),
    nn.Tanh(),
    nn.Unflatten(1, (1, 28, 28)),
)

D_gan = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 1024),
    nn.LeakyReLU(0.2),
    nn.Dropout(0.3),
    nn.Linear(1024, 512),
    nn.LeakyReLU(0.2),
    nn.Dropout(0.3),
    nn.Linear(512, 256),
    nn.LeakyReLU(0.2),
    nn.Dropout(0.3),
    nn.Linear(256, 1),
)
G_gan = G_gan.to(dev)
D_gan = D_gan.to(dev)

G_dc = nn.Sequential(
    nn.Unflatten(1, (NZ, 1, 1)),
    nn.ConvTranspose2d(NZ, 256, 7, 1, 0, bias=False),
    nn.BatchNorm2d(256),
    nn.ReLU(),
    nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
    nn.BatchNorm2d(128),
    nn.ReLU(),
    nn.ConvTranspose2d(128, 1, 4, 2, 1),
    nn.Tanh(),
)
D_dc = nn.Sequential(
    nn.Conv2d(1, 64, 4, 2, 1),
    nn.LeakyReLU(0.2),
    nn.Conv2d(64, 128, 4, 2, 1, bias=False),
    nn.BatchNorm2d(128),
    nn.LeakyReLU(0.2),
    nn.Conv2d(128, 1, 7, 1, 0),
    nn.Flatten(),
)
G_dc = G_dc.to(dev)
D_dc = D_dc.to(dev)


def dcgan_init(module):
    is_conv = isinstance(module, (nn.Conv2d, nn.ConvTranspose2d))
    if is_conv:
        nn.init.normal_(module.weight, 0, 0.02)


G_dc.apply(dcgan_init)
D_dc.apply(dcgan_init)

unet = UNet2DModel(
    sample_size=28,
    in_channels=1,
    out_channels=1,
    layers_per_block=2,
    block_out_channels=(32, 64, 64),
    down_block_types=("DownBlock2D", "DownBlock2D", "DownBlock2D"),
    up_block_types=("UpBlock2D", "UpBlock2D", "UpBlock2D"),
).to(dev)

scheduler = DDPMScheduler(
    num_train_timesteps=T,
    beta_schedule="linear",
    beta_start=1e-4,
    beta_end=0.02,
)


@torch.no_grad()
def ddpm_sample(net, n, record=False):
    net.eval()
    x = torch.randn(n, 1, 28, 28, device=dev)
    snapshots = [x.clone()]

    for t in scheduler.timesteps:
        predicted_noise = net(x, t).sample
        x = scheduler.step(predicted_noise, t, x).prev_sample

        if record and t % 200 == 0:
            snapshots.append(x.clone())

    return x.clamp(-1, 1), snapshots